# 02. 다층 supervisor 실습

목표: deterministic fast-path와 mock vision 판정기를 cascade로 결합하고 항상 같은 structured schema를 반환합니다. mock은 실제 vision model이 아닙니다.

In [ ]:
from dataclasses import asdict, dataclass
from typing import Literal
import json

Status = Literal["completed", "not_completed", "unknown", "needs_review"]

@dataclass
class Decision:
    schema_version: str
    status: Status
    confidence: float
    evidence: list[str]
    advice: str
    evaluator: str

def deterministic_check(observation: dict) -> Decision | None:
    state = observation.get("dom", {}).get("saved_state")
    if state is True:
        return Decision("1.0", "completed", 1.0, ["trusted saved state"], "다음 단계로 이동하세요.", "dom-v1")
    if state is False:
        return Decision("1.0", "not_completed", 1.0, ["trusted unsaved state"], "먼저 저장하세요.", "dom-v1")
    return None

def mock_vision_check(observation: dict) -> Decision:
    # 실제 image 대신 test 가능한 synthetic feature를 사용합니다.
    visual = observation.get("visual_features", {})
    confidence = float(visual.get("confidence", 0.0))
    if confidence < 0.70:
        return Decision("1.0", "unknown", confidence, ["low visual confidence"], "화면을 다시 캡처하세요.", "mock-vision-v1")
    completed = bool(visual.get("success_banner_visible"))
    status: Status = "completed" if completed else "not_completed"
    return Decision("1.0", status, confidence, ["visual success banner check"], "확인했습니다." if completed else "성공 메시지를 확인하세요.", "mock-vision-v1")

def supervise(observation: dict) -> Decision:
    fast = deterministic_check(observation)
    return fast if fast is not None else mock_vision_check(observation)

In [ ]:
observations = [
    {"dom": {"saved_state": True}},
    {"dom": {}, "visual_features": {"success_banner_visible": True, "confidence": 0.91}},
    {"dom": {}, "visual_features": {"success_banner_visible": False, "confidence": 0.45}},
]

for observation in observations:
    decision = supervise(observation)
    payload = asdict(decision)
    assert payload["status"] in {"completed", "not_completed", "unknown", "needs_review"}
    assert 0.0 <= payload["confidence"] <= 1.0
    print(json.dumps(payload, ensure_ascii=False))

## 확장 과제

실제 model adapter를 만들 때도 `Decision` contract는 유지하세요. image redaction, timeout, schema validation과 prompt injection test를 adapter 바깥의 policy layer에서 추가합니다.